# Alert 생성기 실험 ① — 모델이 고른 거래: 묶기 전 vs 묶은 뒤

이진 모델이 의심이라고 한 거래가 Alert 생성기를 거치면서 **무엇이 떨어지고 무엇이 들어오는지**를 정답 라벨로 잰다.

| 단계 | 고른 거래 | 서비스 코드 |
|---|---|---|
| ① 모델 씨앗 | 점수 ≥ 의심 기준 | `alert_pipeline.save_alerts` 의 씨앗 수집 |
| ② 씨앗 보강 | ① + 씨앗과 이어진 낮은 점수 거래 | `alerts/seed_promotion.promote` |
| ③ 묶은 뒤 | Alert 에 들어간 거래 (씨앗 + 연결 거래) | `alerts/block_graph.build_block_graphs` |

- 생성기는 서비스 저장소(`money_laundry`, dev)의 코드를 그대로 불러온다. 복사본이 아니다.
- **서비스와 다른 점**: 서비스는 매일 이어서 돌고 이미 공개된 Alert 를 고정한다. 여기서는 평가 구간의 씨앗을 한 번에 넣고 한 번 돌린다.
- 평가 구간 앞 `PAD_DAYS` 일의 거래는 생성기가 구조를 볼 때만 쓴다. 씨앗과 지표는 평가 구간 안의 거래만이다.
- 처음에는 `DATASET = 'HI-Small'` 로 끝까지 한 번 돌려 본다.

## 0. 설정

In [1]:
DATASET = 'HI-Medium'            # 'HI-Medium' / 'HI-Small' / 'HI-Large'

import json
from pathlib import Path
import gen_lib as GL

# 데이터셋마다 쓰는 점수 파일: (폴더, 모델 이름)
SOURCES = {
    'HI-Medium': ('../../parquets/HI-Medium/service_scores', 'multi280_hgb36_v1'),          # 서비스 이진 모델. 00_service_scores.ipynb 가 만든다
    'HI-Small':  ('../binary_classification/results/HI-Small/model_compare', 'trans_s1'),   # 21_model_compare
    'HI-Large':  ('../binary_classification/results/HI-Large/gnn_structure', 'rev_port'),   # 09_gnn_structure
}
EXP, MODEL = Path(SOURCES[DATASET][0]), SOURCES[DATASET][1]
print(sorted(p.name for p in EXP.glob('*_test_scores.parquet')))   # 여기에 파일이 보여야 한다

S = GL.score_settings(EXP, MODEL)
SCORES, THRESHOLD = S['scores'], S['threshold']             # 의심 기준. 서비스 모델은 서비스 값, 나머지는 그 모델이 검증 구간에서 고른 값
SEED_LOW, CHAIN_FLOOR = S['seed_low'], S['chain_floor']     # 씨앗 보강 값. 서비스 모델 점수에만 있다(없으면 None = 보강 안 함). 끄려면 None, None 으로 바꾼다
print(SCORES, THRESHOLD, SEED_LOW, CHAIN_FLOOR)

PAD_DAYS = 14                    # 평가 구간 앞으로 더 읽는 날짜 수. 생성기의 블록 기간 상한(336시간)과 같다
BUDGET_SCALE = 1                 # 생성기 탐색 예산 배수. 5절에서 예산 초과가 나면 올린다

['multi280_hgb36_v1_test_scores.parquet']
../../parquets/HI-Medium/service_scores/multi280_hgb36_v1_test_scores.parquet 0.194997844 0.1 0.01


`THRESHOLD`, `SEED_LOW`, `CHAIN_FLOOR` 는 **서비스 이진 모델의 점수 척도**에 맞춘 값이다.
다른 모델의 점수 파일을 넣으면 그 모델의 기준으로 바꾸고, 보강 값이 없으면 `None, None` 으로 둔다.

In [2]:
from pathlib import Path
import json, subprocess, time
import duckdb, numpy as np, pandas as pd
import gen_lib as GL

# 평가 구간. 점수 파일에 적혀 있으면 그 구간(00번이 정한 것), 없으면 이진분류 실험의 테스트 구간 (sampling_lib.TEST, bc_lib.WINDOWS 와 같은 값)
TEST_WINDOW = tuple(S['eval_window'] or {'HI-Large': ('2022-10-17', '2022-10-30'), 'HI-Small': ('2022-09-09', '2022-09-10')}[DATASET])
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'parquets').is_dir() or (p / 'datasets').is_dir()), None)
assert ROOT is not None, 'parquets/ 폴더를 못 찾음. 데이터가 있는 GPU 서버의 저장소 안에서 열어야 한다'
BASE = ROOT / 'parquets' / DATASET
OUT = Path.cwd() / 'results' / DATASET / 'before_after_grouping'
OUT.mkdir(parents=True, exist_ok=True)
assert SCORES is not None, '0절의 SCORES 에 점수 파일 경로를 넣는다'
print(BASE, '| 평가 구간', TEST_WINDOW)

/home/donghyeon/workspace/money_laundry_ai_/parquets/HI-Medium | 평가 구간 ('2022-09-08', '2022-09-16')


## 1. 생성기 불러오기

In [3]:
WORKER = GL.find_worker()
G = GL.load_generator(WORKER)
POLICY = G.BlockPolicy()
print(WORKER)
print('정책', POLICY.version, '| 블록 기간 상한', POLICY.span_max_us // 3_600_000_000, '시간 | 허브 기준', POLICY.hub_min_activity, '건')
# 어느 커밋의 생성기로 돌렸는지 남긴다
COMMIT = subprocess.run(['git', '-C', str(WORKER), 'log', '-1', '--format=%h %ad %s', '--date=short'], capture_output=True, text=True).stdout.strip()
print(COMMIT)

/home/donghyeon/workspace/money_laundry/backend/api-server/worker
정책 block-pin-3 | 블록 기간 상한 336 시간 | 허브 기준 512 건
ce5b429 2026-11-19 Merge pull request #227 from dh0728/backend


## 2. 거래 읽기
평가 구간과 그 앞 `PAD_DAYS` 일을 시간순으로 읽는다.

In [4]:
con = duckdb.connect()
a, b = TEST_WINDOW
start = (pd.Timestamp(a) - pd.Timedelta(days=PAD_DAYS)).date()
t = time.perf_counter()
tx = con.execute(f"""
    SELECT edge_id, ts, src, dst, paid, payment_currency, CAST(is_laundering AS TINYINT) AS y, day
    FROM read_parquet('{BASE / 'transactions.parquet'}')
    WHERE day BETWEEN DATE '{start}' AND DATE '{b}'
    ORDER BY ts, edge_id""").df()
print(f'{len(tx):,} 건  {tx.ts.min()} ~ {tx.ts.max()}  ({time.perf_counter() - t:.0f}s)')
tx.head(3)

31,891,231 건  2022-09-01 00:00:00 ~ 2022-09-16 23:59:00  (9s)


,edge_id,ts,src,dst,paid,payment_currency,y,day
0,1,2022-09-01,3,3,19.18,US Dollar,0,2022-09-01
1,2,2022-09-01,25,77460,17955.11,US Dollar,0,2022-09-01
2,3,2022-09-01,31,84585,1199.83,US Dollar,0,2022-09-01


### 2-1. USD 환산
생성기는 금액 비율로 흐름을 잇는다. 서비스와 같은 고정 환율로 보낸 금액을 USD 로 바꾼다.

In [5]:
tx['usd'] = GL.to_usd(tx.paid, tx.payment_currency)
print('환율을 모르는 통화:', list(tx.loc[tx.usd.isna(), 'payment_currency'].unique()))   # 비어 있어야 정상
tx.groupby('payment_currency').agg(n=('usd', 'size'), usd_median=('usd', 'median'))

환율을 모르는 통화: []


,n,usd_median
payment_currency,,
Australian Dollar,918520,878.206399
Bitcoin,688900,852.614068
Brazil Real,629509,890.838577
Canadian Dollar,1078330,849.791550
Euro,7313735,878.579798
Mexican Peso,841256,907.083558
Ruble,975263,891.538636
Rupee,738517,875.148933
Saudi Riyal,564502,899.984014


### 2-2. 패턴 라벨 붙이기
세탁 거래 중 Patterns 파일에 없는 것은 `NO_PATTERN` 으로 둔다.

In [6]:
pat = con.execute(f"""SELECT edge_id, any_value(pattern_type) AS pattern_type
                      FROM read_parquet('{BASE / 'patterns.parquet'}') WHERE edge_id IS NOT NULL GROUP BY edge_id""").df()
n_before = len(tx)
tx = tx.merge(pat, on='edge_id', how='left')
assert len(tx) == n_before and tx.ts.is_monotonic_increasing        # 행 수와 순서가 그대로여야 한다
tx['pattern_type'] = np.where(tx.y == 1, tx.pattern_type.fillna('NO_PATTERN'), '')
in_eval = ((pd.to_datetime(tx.day) >= pd.Timestamp(a)) & (pd.to_datetime(tx.day) <= pd.Timestamp(b))).to_numpy()
print(f'평가 구간 거래 {in_eval.sum():,} | 그중 세탁 {int(tx.y[in_eval].sum()):,}')
tx[in_eval & (tx.y == 1).to_numpy()].pattern_type.value_counts()

평가 구간 거래 16,955,723 | 그중 세탁 19,786


pattern_type
NO_PATTERN        7139
GATHER-SCATTER    2312
SCATTER-GATHER    2301
STACK             2162
FAN-IN            1324
CYCLE             1210
BIPARTITE         1197
FAN-OUT           1159
RANDOM             982
Name: count, dtype: int64

## 3. 모델 점수 → 씨앗 (①)

In [7]:
scores = pd.read_parquet(SCORES)          # edge_id, y, score
n_before = len(tx)
tx = tx.merge(scores[['edge_id', 'score']].drop_duplicates('edge_id'), on='edge_id', how='left')
assert len(tx) == n_before and tx.ts.is_monotonic_increasing
score = tx.score.to_numpy(dtype='float64')
print(f'점수 {len(scores):,} 건 | 평가 구간인데 점수가 없는 거래 {int((in_eval & np.isnan(score)).sum()):,} (0 이어야 정상)')
# 점수 파일의 라벨과 거래 파일의 라벨이 같은지
check = scores.merge(tx[['edge_id', 'y']], on='edge_id', suffixes=('_score', '_tx'))
print('라벨 불일치:', int((check.y_score != check.y_tx).sum()))

점수 16,955,723 건 | 평가 구간인데 점수가 없는 거래 0 (0 이어야 정상)
라벨 불일치: 0


In [8]:
y = tx.y.to_numpy()
pattern_type = tx.pattern_type.to_numpy()
pattern_mask = (y == 1) & (pattern_type != 'NO_PATTERN')
seed_model = in_eval & (np.nan_to_num(score, nan=-1.0) >= THRESHOLD)
stages = [GL.stage_metrics('① 모델 씨앗', seed_model, y, in_eval, pattern_mask)]
pd.DataFrame(stages)

,단계,고른_거래,세탁,정상,정밀도,재현율_전체,재현율_패턴만
0,① 모델 씨앗,13343,12041,1302,0.9024,0.6086,0.9475


## 4. 씨앗 보강 (②)
기준 미만이지만 씨앗과 계좌를 공유하거나(점수 ≥ `SEED_LOW`) 금액이 이어지는(점수 ≥ `CHAIN_FLOOR`) 거래를 씨앗에 더한다.

In [9]:
t = time.perf_counter()
edges = GL.make_edges(G, tx)
print(f'생성기 입력 {len(edges):,} 건 ({time.perf_counter() - t:.0f}s)')

생성기 입력 31,891,231 건 (2s)


In [10]:
if SEED_LOW is None or CHAIN_FLOOR is None:
    promoted = np.array([], dtype=np.int64)
else:
    t = time.perf_counter()
    promoted = GL.promote_seeds(G, edges, score, np.flatnonzero(seed_model), SEED_LOW, CHAIN_FLOOR)
    print(f'보강으로 들어온 거래 {len(promoted):,} 건 ({time.perf_counter() - t:.0f}s)')
seed_all = seed_model.copy()
seed_all[promoted] = True
stages.append(GL.stage_metrics('② 씨앗 보강', seed_all, y, in_eval, pattern_mask))
pd.DataFrame(stages)

보강으로 들어온 거래 876 건 (1s)


,단계,고른_거래,세탁,정상,정밀도,재현율_전체,재현율_패턴만
0,① 모델 씨앗,13343,12041,1302,0.9024,0.6086,0.9475
1,② 씨앗 보강,14219,12412,1807,0.8729,0.6273,0.9746


In [11]:
# 보강으로 들어온 거래가 실제로 무엇이었나
is_promoted = np.zeros(len(tx), dtype=bool)
is_promoted[promoted] = True
GL.split_by_label(is_promoted, y, pattern_type)

정상                505
STACK              96
GATHER-SCATTER     56
SCATTER-GATHER     53
CYCLE              47
RANDOM             38
FAN-IN             30
NO_PATTERN         28
FAN-OUT            20
BIPARTITE           3
Name: count, dtype: int64

## 5. 생성기 실행
씨앗을 한 번에 넣고 묶는다. HI-Large 는 색인을 만드는 데 몇 분 걸릴 수 있다.

In [12]:
try:
    batch, info = GL.run_generator(G, edges, np.flatnonzero(seed_all), budget_scale=BUDGET_SCALE)
except G.GraphBudgetExceeded as error:
    raise RuntimeError(f'생성기 탐색 예산 초과: {error}. 0절의 BUDGET_SCALE 을 올려 다시 돌린다') from error
info

{'policy': 'block-pin-3',
 'index_seconds': 66.1,
 'build_seconds': 6.8,
 'edge_visits': 1360537,
 'graphs': 2051,
 'unassigned': 3344}

In [13]:
role, graph, reason, alerts = GL.batch_table(batch, len(tx))
print(f'Alert {len(alerts):,} 개 | Alert 에 못 들어간 씨앗 {len(batch.unassigned):,} 건')
batch.stats

Alert 2,051 개 | Alert 에 못 들어간 씨앗 3,344 건


{'adopted_BRIDGE': 329,
 'adopted_CLOSURE': 83,
 'adopted_PASS': 3229,
 'adopted_RELAY': 99,
 'adopted_STAR_IN': 567,
 'adopted_STAR_OUT': 531,
 'bridges': 545,
 'face_intervals': 18433,
 'graphs': 2051,
 'pass_adopted': 3229,
 'pass_candidates': 5238,
 'pass_contested': 2009,
 'relays': 100,
 'relays_rejected': 1,
 'repeat_pairs_skipped': 1136,
 'stars': 1235,
 'stars_extend': 0,
 'stars_rejected': 137,
 'unassigned': 3344,
 'walk_truncated': 383}

## 6. 세 단계 비교 (③)

In [14]:
member = role > GL.ROLE_NONE
print('Alert 구성 거래 중 평가 구간 밖(앞쪽 PAD 구간) 거래:', int((member & ~in_eval).sum()), '건 → 아래 지표에서는 뺀다')
stages.append(GL.stage_metrics('③ 묶은 뒤', member, y, in_eval, pattern_mask))
result = pd.DataFrame(stages)
result

Alert 구성 거래 중 평가 구간 밖(앞쪽 PAD 구간) 거래: 636 건 → 아래 지표에서는 뺀다


,단계,고른_거래,세탁,정상,정밀도,재현율_전체,재현율_패턴만
0,① 모델 씨앗,13343,12041,1302,0.9024,0.6086,0.9475
1,② 씨앗 보강,14219,12412,1807,0.8729,0.6273,0.9746
2,③ 묶은 뒤,10890,10370,520,0.9522,0.5241,0.8178


**읽는 법**
- ① → ③ 에서 `정상` 이 줄어든 만큼이 생성기가 걸러낸 오탐이다.
- ① → ③ 에서 `세탁` 이 줄어든 만큼이 생성기 때문에 놓친 세탁이다.
- `재현율_패턴만` 은 Patterns 파일에 있는 세탁만 분모로 한 값이다. 무패턴 제외 기준으로 볼 때 쓴다.

## 7. 무엇이 떨어지고 무엇이 들어왔나

In [15]:
# 7-1. Alert 에 못 들어간 씨앗: 탈락 사유 × 실제 라벨
dropped = seed_all & (role == GL.ROLE_NONE)
label = np.where(y == 1, pattern_type, '정상')
pd.crosstab(pd.Series(reason[dropped], name='탈락 사유'), pd.Series(label[dropped], name='실제'), margins=True)

실제,BIPARTITE,CYCLE,FAN-IN,FAN-OUT,GATHER-SCATTER,NO_PATTERN,RANDOM,SCATTER-GATHER,STACK,정상,All
탈락 사유,,,,,,,,,,,
COMPONENT_SPAN,0,1,0,0,0,0,0,0,0,0,1
CORE_CAP,13,6,15,17,58,0,7,26,14,14,170
LINK_ONLY,33,30,14,23,28,9,15,37,38,60,287
NO_STRUCTURE,923,78,31,36,31,50,51,21,413,1205,2839
SELF_TRANSFER,0,0,0,0,33,0,0,0,0,14,47
All,969,115,60,76,150,59,73,84,465,1293,3344


In [16]:
# 7-2. 씨앗이 아닌데 구조상 Alert 에 들어온 거래(연결 거래)
GL.split_by_label(role == GL.ROLE_CONNECTION, y, pattern_type)

SCATTER-GATHER    362
CYCLE             179
GATHER-SCATTER     29
STACK              18
RANDOM             17
FAN-IN             16
정상                 12
FAN-OUT            10
BIPARTITE           8
Name: count, dtype: int64

In [17]:
# 7-3. 패턴별 재현율이 단계마다 어떻게 바뀌나
rows = []
for kind in sorted(set(pattern_type[in_eval & (y == 1)])):
    base = in_eval & (y == 1) & (pattern_type == kind)
    rows.append(dict(패턴=kind, 세탁_거래=int(base.sum()),
                     모델_씨앗=round((base & seed_model).sum() / base.sum(), 4),
                     씨앗_보강=round((base & seed_all).sum() / base.sum(), 4),
                     묶은_뒤=round((base & member).sum() / base.sum(), 4)))
by_pattern = pd.DataFrame(rows)
by_pattern

,패턴,세탁_거래,모델_씨앗,씨앗_보강,묶은_뒤
0,BIPARTITE,1197,0.9148,0.9173,0.1078
1,CYCLE,1210,0.9504,0.9893,0.8959
2,FAN-IN,1324,0.9426,0.9653,0.9199
3,FAN-OUT,1159,0.9681,0.9853,0.9198
4,GATHER-SCATTER,2312,0.9550,0.9792,0.9144
5,NO_PATTERN,7139,0.0081,0.0120,0.0038
6,RANDOM,982,0.9532,0.9919,0.9175
7,SCATTER-GATHER,2301,0.9674,0.9904,0.9570
8,STACK,2162,0.9241,0.9685,0.7535


In [18]:
# 7-4. 시도가 평가 구간 뒤로 이어지는지에 따라 나눠 본다
#   구간 안에 끝남   = 그 시도의 마지막 거래가 평가 구간 마지막 날 이전
#   구간 뒤로 이어짐 = 마지막 거래가 그 뒤. 구간 안에서는 패턴이 완성되기 전 모습만 보인다
tail = GL.attempt_tail(con, BASE / 'patterns.parquet', b).set_index('edge_id')
attempt_id = tx.edge_id.map(tail.attempt_id).to_numpy()
ends = tx.edge_id.map(tail.after).map(GL.TAIL_LABELS).to_numpy()
rows = []
for kind in [*sorted(set(pattern_type[in_eval & pattern_mask])), '(패턴 전체)']:
    for label in GL.TAIL_LABELS.values():
        base = in_eval & pattern_mask & (ends == label) & ((pattern_type == kind) | (kind == '(패턴 전체)'))
        if base.sum():
            rows.append(dict(패턴=kind, 끝=label, 시도=len(set(attempt_id[base])), 세탁_거래=int(base.sum()),
                             모델_씨앗=round((base & seed_model).sum() / base.sum(), 4),
                             씨앗_보강=round((base & seed_all).sum() / base.sum(), 4),
                             묶은_뒤=round((base & member).sum() / base.sum(), 4)))
by_tail = pd.DataFrame(rows)
by_tail

,패턴,끝,시도,세탁_거래,모델_씨앗,씨앗_보강,묶은_뒤
0,BIPARTITE,구간 안에 끝남,218,1099,0.9172,0.9199,0.1156
1,BIPARTITE,구간 뒤로 이어짐,23,98,0.8878,0.8878,0.0204
2,CYCLE,구간 안에 끝남,201,847,0.9481,0.9906,0.9067
3,CYCLE,구간 뒤로 이어짐,107,363,0.9559,0.9862,0.8705
4,FAN-IN,구간 안에 끝남,195,957,0.9467,0.9655,0.9216
5,FAN-IN,구간 뒤로 이어짐,85,367,0.9319,0.9646,0.9155
6,FAN-OUT,구간 안에 끝남,192,837,0.9749,0.9940,0.9223
7,FAN-OUT,구간 뒤로 이어짐,76,322,0.9503,0.9627,0.9130
8,GATHER-SCATTER,구간 안에 끝남,126,868,0.9724,0.9908,0.8733
9,GATHER-SCATTER,구간 뒤로 이어짐,163,1444,0.9446,0.9723,0.9391


## 8. Alert 단위로 보기
"맞은 Alert" 의 기준은 아직 정하지 않았다. 세탁 거래가 1건 이상 / 절반 이상 / 전부인 Alert 수를 같이 본다.

In [19]:
alerts = GL.alert_purity(graph, role, y, alerts)
summary = dict(Alert_수=len(alerts), 세탁_1건_이상=int((alerts.laundering >= 1).sum()),
               세탁_절반_이상=int((alerts.purity >= 0.5).sum()), 전부_세탁=int((alerts.purity == 1).sum()),
               전부_정상=int((alerts.laundering == 0).sum()))
print(summary)
alerts.members.describe(percentiles=[.5, .9, .99]).round(1)

{'Alert_수': 2051, '세탁_1건_이상': 2007, '세탁_절반_이상': 1998, '전부_세탁': 1739, '전부_정상': 44}


count    2051.0
mean        5.6
std         7.1
min         2.0
50%         2.0
90%        12.0
99%        35.0
max        64.0
Name: members, dtype: float64

In [20]:
# 어떤 구조 근거로 묶였나 (STAR_IN, STAR_OUT, RELAY, PASS, PATH, BRIDGE, ...)
alerts.groupby('kinds').agg(n=('alert', 'size'), purity_mean=('purity', 'mean'), members_mean=('members', 'mean')).sort_values('n', ascending=False).head(15).round(3)

,n,purity_mean,members_mean
kinds,,,
PASS,1156,0.964,2.719
STAR_IN,292,0.950,6.236
STAR_OUT,243,0.926,5.243
BRIDGE PASS STAR_IN STAR_OUT,102,0.915,19.814
CLOSURE PASS,45,0.993,8.133
PASS STAR_OUT,38,0.872,6.211
BRIDGE STAR_IN,33,0.881,12.000
PASS RELAY STAR_IN STAR_OUT,31,0.967,16.548
RELAY STAR_IN STAR_OUT,20,0.993,13.200


## 9. 저장

In [21]:
result.to_csv(OUT / 'stages.csv', index=False)
by_pattern.to_csv(OUT / 'by_pattern.csv', index=False)
by_tail.to_csv(OUT / 'by_pattern_tail.csv', index=False)
alerts.to_csv(OUT / 'alerts.csv', index=False)
keep = seed_all | member
pd.DataFrame(dict(edge_id=tx.edge_id[keep], y=y[keep], pattern_type=pattern_type[keep], score=score[keep],
                  seed_model=seed_model[keep], promoted=is_promoted[keep], role=role[keep], alert=graph[keep],
                  reason=reason[keep])).to_parquet(OUT / 'members.parquet', index=False)
json.dump(dict(dataset=DATASET, scores=str(SCORES), threshold=THRESHOLD, seed_low=SEED_LOW, chain_floor=CHAIN_FLOOR,
               pad_days=PAD_DAYS, test_window=list(TEST_WINDOW), generator_commit=COMMIT, info=info, alert_summary=summary),
          open(OUT / 'run.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(sorted(p.name for p in OUT.iterdir()))

['alerts.csv', 'by_pattern.csv', 'by_pattern_tail.csv', 'members.parquet', 'run.json', 'stages.csv']
